# 如何创建和查询向量存储
[API](https://python.langchain.com/api_reference/index.html)

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv("apikey.env")
BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')
deepseek_chat_model = 'deepseek-chat'
if  not API_KEY:
    raise ValueError("WARNING: NOT FOUND OPENAI_API_KEY，PLEASE CHECK .env SETING。")
else:
    print("SECESSFULLY!")

model = ChatDeepSeek(api_key=API_KEY, base_url=BASE_URL, model=deepseek_chat_model)
embeddings = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")

SECESSFULLY!


C:\Users\hhm18\AppData\Local\Temp\ipykernel_15100\3288782776.py:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")


对于**非结构化数据**的存储和搜索可以将其**embedding**，并且存储如**向量数据库**，查询则使用向量的**相似度**进行查找匹配。

## 开始
有许多优秀的、开源的向量存储选项（Chroma， FAISS， LanceDB...）

这里我们使用FAISS：`pip install faiss-cpu`

首先我们加载数据（并分块）

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

# Load the document, split it into chunks, 
# embed each chunk and load it into the vector store.
raw_documents = TextLoader('./state2/example/TestVectorBD.txt', encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=10)
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 244, which is longer than the specified 200
Created a chunk of size 237, which is longer than the specified 200
Created a chunk of size 466, which is longer than the specified 200
Created a chunk of size 329, which is longer than the specified 200
Created a chunk of size 327, which is longer than the specified 200
Created a chunk of size 286, which is longer than the specified 200
Created a chunk of size 269, which is longer than the specified 200
Created a chunk of size 298, which is longer than the specified 200
Created a chunk of size 409, which is longer than the specified 200
Created a chunk of size 304, which is longer than the specified 200
Created a chunk of size 222, which is longer than the specified 200


In [2]:
len(documents)

58

In [5]:
from langchain_community.vectorstores import FAISS
db = FAISS.from_documents(documents, embeddings)

## 相似性搜索
所有向量存储都暴露了 `similarity_search` 方法。 
- 这将接收传入的文档，创建它们的嵌入，然后找到所有**具有最相似嵌入**的文档。

In [7]:
query = "什么是注意力以及注意力的缺失会导致什么问题？"
docs = db.similarity_search(query)
print(f"总片段有：{len(documents)}段, \n\n搜索片段：{len(docs)}段, \n\n展示第一个片段:\n{docs[0].page_content}")

总片段有：58段, 

搜索片段：4段, 

展示第一个片段:
这并非个人的意志力缺陷，而是一个时代的症候。我们正漂浮在一片无边无际的信息汪洋之中，而指引我们航向的内心灯塔——注意力，却正变得摇曳不定、光芒涣散。这片汪洋表面波澜壮阔，内里却暗流汹涌，它重塑着我们的认知习惯、情感模式乃至社会结构。本文旨在深入这片汪洋，探寻其成因，剖析其影响，并最终追问：在这个注意力被极度稀释的时代，我们如何能够重新集结精神的能量，为漂泊的心灵重建一座稳固而明亮的家园？


### 按向量进行相似性搜索

In [ ]:
embedding_vector = embeddings.embed_query(query)
docs = db.similarity_search_by_vector(embedding_vector)
print(f"总片段有：{len(documents)}段, \n\n搜索片段：{len(docs)}段, \
      \n\n展示第一个片段:\n{docs[0].page_content}")

总片段有：58段, 

搜索片段：4段, 

展示第一个片段:
这并非个人的意志力缺陷，而是一个时代的症候。我们正漂浮在一片无边无际的信息汪洋之中，而指引我们航向的内心灯塔——注意力，却正变得摇曳不定、光芒涣散。这片汪洋表面波澜壮阔，内里却暗流汹涌，它重塑着我们的认知习惯、情感模式乃至社会结构。本文旨在深入这片汪洋，探寻其成因，剖析其影响，并最终追问：在这个注意力被极度稀释的时代，我们如何能够重新集结精神的能量，为漂泊的心灵重建一座稳固而明亮的家园？


## 支持异步操作

LangChain支持在向量存储上进行异步操作。所有方法都可以使用其异步对应方法调用，前缀为a，表示async。

In [9]:
docs = await db.asimilarity_search(query)
docs

[Document(id='b5a3da28-2516-4e9f-aa0d-16290cff401c', metadata={'source': './state2/example/TestVectorBD.txt'}, page_content='这并非个人的意志力缺陷，而是一个时代的症候。我们正漂浮在一片无边无际的信息汪洋之中，而指引我们航向的内心灯塔——注意力，却正变得摇曳不定、光芒涣散。这片汪洋表面波澜壮阔，内里却暗流汹涌，它重塑着我们的认知习惯、情感模式乃至社会结构。本文旨在深入这片汪洋，探寻其成因，剖析其影响，并最终追问：在这个注意力被极度稀释的时代，我们如何能够重新集结精神的能量，为漂泊的心灵重建一座稳固而明亮的家园？'),
 Document(id='49693ee3-d227-4748-94cd-afc18b72eda9', metadata={'source': './state2/example/TestVectorBD.txt'}, page_content='**第二部分：涣散的心智——注意力失焦的心理与社会后果**\n\n当我们的注意力长期处于被切割、被劫持的状态时，其影响会从认知层面，逐渐渗透到心理、情感乃至社会文化的肌理之中。\n\n**一、认知的磨损：从深度思考到“肤浅”浏览**'),
 Document(id='c4d0903c-0448-4f89-a22a-89fd6b8c8a1d', metadata={'source': './state2/example/TestVectorBD.txt'}, page_content='*   **“连续部分注意力”状态：** 我们常常以为自己是在“多任务处理”，但大脑实际上是在多个任务之间进行快速的、高耗能的切换。这种状态被称为“连续部分注意力”（Continuous Partial Attention），它使我们长期处于一种低水平的警觉和焦虑中，无法对任何一件事物进行深入的投入。其结果是，我们似乎接触了很多信息，但很少能形成系统、深刻的知识体系。\n*   **批判性思维的退化：** 深度阅读和线性思维是培养批判性思维的基础。当我们习惯了快餐式的信息消费，我们便倾向于接受简单化、情绪化的结论，而非自己去探究复杂的事实、辨析论证的漏洞。我们更可能被煽动性的标题和立场鲜明的口号所吸引，而

## 简单完整示例

```md
graph LR
A[用户问题] --> B(向量数据库检索)
B --> C{相关文档 chunks}
C --> D[LLM + Prompt]
D --> E[生成最终答案]```